# Tahap 3: Adaptasi melalui Fine-Tuning LoRA
Subbab 3.4, 3.6.7, dan 3.6.8, Gambar 4.8. Varian terpilih Tahap 1 dilatih dengan LoRA pada data latih, checkpoint dipilih dengan WER validasi, lalu model digabung, dikonversi, dan diuji atas data uji. Hasilnya dibandingkan dengan T1-E dan dengan T2 (Tabel 3.7) untuk rumusan masalah kedua.

```
[1] Pelatihan LoRA                -> latih/ (konfigurasi, log_langkah, log_validasi, validasi_segmen, sesi, adapter_terbaik/)
[2] Penggabungan dan konversi     -> model/ (hf_gabung, ct2_t3, kontrol: hf_dasar, ct2_dasar), konversi.json
[3] Pemeriksaan kesetaraan        -> kesetaraan/ (hipotesis tiap model, kesetaraan.csv)
[4] Rekaman validasi utuh         -> validasi_utuh/ (mentah, log, metrik_rekaman.csv)
[5] Transkripsi T3                -> mentah/<ID>.json, waktu_transkripsi.jsonl
[6] Metrik dan log per data       -> metrik_blok.csv, metrik_t3.csv, rtf_ulangan.csv, verifikasi_jiwer.csv, log/
[7] Analisis terhadap T1-E dan T2 -> statistik_deskriptif.csv, ambang.csv, wilcoxon.csv, perubahan_rekaman.csv, per_jalur.csv,
                                     perubahan_istilah.csv, kesalahan_istilah.csv, kandidat_halusinasi.csv,
                                     pelatihan_per_epoch.csv, kurva_per_epoch.png, konvergensi.csv
[8] Sumber daya                   -> sumber_daya.csv
```

Notebook ini menjalankan seluruh Tahap 3 di perangkat Tabel 3.2. Untuk pelatihan dijalankan di server, digunakan notebook `11a_tahap3_latih.ipynb` di server lalu `11b_tahap3_evaluasi.ipynb` di laptop. Isi keduanya sama dengan notebook ini.

## 0. Persiapan

In [ ]:
from pathlib import Path
import sys, os, json, math, datetime, importlib, subprocess, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
import src.latih_lora as ll
import src.kurva_pelatihan as kp
import src.transkripsi as tr
import src.metrik as mt
import src.log_data as ld
import src.analisis as an
for m in (pt, ll, kp, tr, mt, ld, an):
    importlib.reload(m)

OUT        = ROOT / "data" / "output"
MANIFEST   = OUT / "manifest_rekaman.csv"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
ACUAN_DIR  = OUT / "praproses_teks" / "acuan"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
SEG_LATIH  = OUT / "segmen_latih.csv"
SEG_AUDIO  = ROOT / "data" / "processed" / "segmen_latih"
T1         = OUT / "tahap1"
T2         = OUT / "tahap2"
T3         = OUT / "tahap3"
LATIH      = T3 / "latih"          # hasil pelatihan; satu-satunya folder yang dibawa pulang dari server
KERJA      = T3 / "_kerja"
STATE      = KERJA / "state"       # state lengkap untuk melanjutkan pelatihan (besar, tidak di-commit)
MODEL      = T3 / "model"          # model hasil gabung dan konversi (besar, tidak di-commit)
MENTAH     = T3 / "mentah"
LOG_WAKTU  = T3 / "waktu_transkripsi.jsonl"
for d in (T3, LATIH, KERJA):
    d.mkdir(parents=True, exist_ok=True)

rel = lambda p: Path(p).relative_to(ROOT).as_posix()

In [ ]:
# Varian terpilih dan konfigurasi Tahap 1
pemilihan = json.loads((T1 / "pemilihan_varian.json").read_text(encoding="utf-8"))
meta_t1 = json.loads((T1 / "run_meta_tahap1.json").read_text(encoding="utf-8"))
VARIAN = pemilihan["terpilih"]
KODE_DASAR, KODE_T2, KODE = pemilihan["kode_baseline"], "T2", "T3"

manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False, encoding="utf-8-sig").set_index("file_id")
uji = sorted(manifest.index[manifest.split == "uji"])
validasi = sorted(manifest.index[manifest.split == "validasi"])
assert len(uji) == 9 and len(validasi) == 4, f"{len(uji)} rekaman uji, {len(validasi)} rekaman validasi"

kamus = pt.muat_kamus(KAMUS_PATH)
assert kamus.terkunci and kamus.tanggal_kunci == meta_t1["kamus"]["tanggal_kunci"], "kamus berbeda dengan Tahap 1"
seg = pd.read_csv(SEG_LATIH, dtype={"seg_latih_id": str}, keep_default_na=False, encoding="utf-8-sig")
assert set(seg.loc[seg.split == "validasi", "file_id"]) == set(validasi), "segmen validasi tidak sama dengan manifest"
assert not set(seg.file_id) & set(uji), "ada segmen latih/validasi dari rekaman uji"
glosarium = pt.muat_glosarium(GLOSARIUM)
kanonik = (pd.read_csv(GLOSARIUM, dtype=str, keep_default_na=False, encoding="utf-8-sig")
             .drop_duplicates("entri_id").set_index("entri_id").bentuk_kanonik.to_dict())

print(f"Varian: {VARIAN}, baseline {KODE_DASAR} dan {KODE_T2}")
print(f"Segmen: {seg.split.value_counts().to_dict()}; rekaman validasi: {validasi}")
print(f"Kamus terkunci sejak {kamus.tanggal_kunci}")

In [ ]:
# WAV segmen tidak ada di GitHub, jadi keberadaan dan formatnya diperiksa dulu.
SPLIT_DIPERIKSA = ['latih', 'validasi']
masalah, beda_durasi = [], []
dicek = seg[seg.split.isin(SPLIT_DIPERIKSA)]
for r in dicek.itertuples():
    p = SEG_AUDIO / r.split / f"{r.seg_latih_id}.wav"
    if not p.exists():
        masalah.append((r.seg_latih_id, "tidak ada"))
        continue
    i = sf.info(str(p))
    if (i.samplerate, i.channels) != (16000, 1):
        masalah.append((r.seg_latih_id, f"{i.samplerate} Hz, {i.channels} kanal"))
    elif abs(i.frames / i.samplerate - float(r.dur)) > 0.1:
        beda_durasi.append((r.seg_latih_id, round(i.frames / i.samplerate, 2), float(r.dur)))
masalah = pd.DataFrame(masalah, columns=["seg_latih_id", "masalah"])
print(f"WAV {'/'.join(SPLIT_DIPERIKSA)}: {len(dicek) - len(masalah)}/{len(dicek)} siap (16 kHz mono)")
if beda_durasi:
    print(f"[!] {len(beda_durasi)} WAV berbeda durasi > 0,1 detik dari segmen_latih.csv:", beda_durasi[:5])
assert masalah.empty, "WAV bermasalah:\n" + masalah.head(20).to_string()

In [ ]:
def nvsmi(q):
    try:
        out = subprocess.check_output(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"],
                                      stderr=subprocess.DEVNULL)
        return out.decode(errors="replace").strip().splitlines()[0].strip()
    except Exception:
        return None

GPU = {"nama": nvsmi("name"), "vram_total_mb": nvsmi("memory.total"),
       "vram_terpakai_mb": nvsmi("memory.used"), "driver": nvsmi("driver_version")}
print(GPU)
if GPU["vram_terpakai_mb"] and int(GPU["vram_terpakai_mb"]) > 400:
    print("[!] VRAM terpakai sebelum mulai lebih dari 400 MB. Alihkan tampilan ke iGPU dan tutup aplikasi lain.")

## 1. Pelatihan LoRA
Konfigurasi Tabel 3.6. Nilai LoRA dan laju belajar dibaca dari uji kelayakan Tahap 1, ukuran batch dan gradient checkpointing dari `pemilihan_varian.json`, lalu akumulasi gradien mengembalikan batch efektif ke 4.
- Satu langkah optimizer berisi 4 segmen. 1.001 segmen latih tidak habis dibagi 4, jadi langkah terakhir tiap epoch berisi sisa segmen dan loss-nya dirata-rata atas jumlah segmen yang sebenarnya.
- WER validasi (greedy, tanpa timestamp, total kesalahan dibagi total kata acuan 149 segmen) dihitung di langkah 0 sebagai pembanding, tiap 200 langkah, dan di langkah terakhir. Langkah 0 tidak ikut dipilih.
- Konfigurasi dikunci di `latih/konfigurasi.json`. Bila diubah setelah pelatihan berjalan, worker menolak melanjutkan.

In [ ]:
EPOCH, BATCH_EFEKTIF, RASIO_PEMANASAN, EVAL_TIAP, SEED = 35, 4, 0.1, 200, 42
LORA = meta_t1["uji_kelayakan"]["lora"]
assert LORA == {"r": 32, "alpha": 64, "dropout": 0.0, "target_modules": ["q_proj", "v_proj"]}, LORA
BATCH = pemilihan["batch_tahap3"]

CFG_LATIH = {
    "nama": KODE, "model_id": f"openai/whisper-{VARIAN}", "lora": LORA,
    "laju_belajar": meta_t1["uji_kelayakan"]["laju_belajar"], "epoch": EPOCH,
    "batch": BATCH, "akumulasi": BATCH_EFEKTIF // BATCH, "rasio_pemanasan": RASIO_PEMANASAN,
    "gradient_checkpointing": pemilihan["gradient_checkpointing"], "seed": SEED,
    "eval_tiap": EVAL_TIAP, "eval_langkah_akhir": True, "eval_batch": 2, "maks_token_baru": 440,
    # nilai bawaan torch.optim.AdamW, sama dengan uji kelayakan
    "adamw": {"betas": [0.9, 0.999], "eps": 1e-8, "weight_decay": 0.01},
    "segmen_latih": rel(SEG_LATIH), "audio_dir": rel(SEG_AUDIO), "kamus": rel(KAMUS_PATH),
    "glosarium": rel(GLOSARIUM), "folder": rel(LATIH), "folder_state": rel(STATE), "perangkat": "cuda",
}
assert BATCH * CFG_LATIH["akumulasi"] == BATCH_EFEKTIF
R = ll.rencana(int((seg.split == "latih").sum()), BATCH, CFG_LATIH["akumulasi"], EPOCH, RASIO_PEMANASAN)
PATH_CFG = KERJA / "latih_lora.json"

print(pd.Series({"model": CFG_LATIH["model_id"], "LoRA": f"r {LORA['r']}, alpha {LORA['alpha']}, dropout {LORA['dropout']}, "
                 f"{'/'.join(LORA['target_modules'])} di encoder dan decoder",
                 "pengoptimal": f"AdamW, lr {CFG_LATIH['laju_belajar']}, linear decay",
                 "batch": f"{BATCH} x akumulasi {CFG_LATIH['akumulasi']} = {BATCH_EFEKTIF}",
                 "gradient checkpointing": CFG_LATIH["gradient_checkpointing"],
                 "langkah optimizer": f"{R['langkah_per_epoch']} per epoch x {EPOCH} = {R['total_langkah']}",
                 "pemanasan": f"{R['langkah_pemanasan']} langkah",
                 "validasi": f"langkah 0, tiap {EVAL_TIAP}, dan langkah {R['total_langkah']} "
                             f"({R['total_langkah'] // EVAL_TIAP + 1} kali dapat dipilih)",
                 "seed": SEED}).to_string())

In [ ]:
JALANKAN_PELATIHAN = True
BATAS_JAM = None   # mis. 8: berhenti rapi pada simpanan pertama setelah 8 jam; jalankan ulang sel ini untuk melanjutkan

if JALANKAN_PELATIHAN:
    print(f"VRAM terpakai sebelum mulai: {nvsmi('memory.used')} MB")
    tr.jalankan_terpisah(CFG_LATIH | {"batas_jam": BATAS_JAM}, PATH_CFG, cwd=ROOT, modul="src.latih_lora")
else:
    print("JALANKAN_PELATIHAN False: memakai hasil yang sudah ada di", rel(LATIH))

### 1.1 Kurva pelatihan
Loss per langkah optimizer dan WER validasi per evaluasi, dibaca dari log. Garis putus-putus adalah WER langkah 0 (model dasar dengan transformers). Dapat dijalankan selama pelatihan berlangsung untuk memantau.

In [ ]:
d = kp.muat(LATIH)
konf, L, V = d["konfigurasi"], d["langkah"], d["validasi"]
total = konf["rencana"]["total_langkah"]
g = int(L.langkah.max()) if len(L) else 0
print(f"Langkah {g}/{total} ({100 * g / total:.1f}%), epoch {g / d['langkah_per_epoch']:.2f}, {len(d['sesi'])} sesi; "
      f"dilewati GradScaler: {int(L.dilewati.sum()) if len(L) else 0}; "
      f"norma gradien tak hingga/NaN: {int(L.norma_gradien.isna().sum()) if len(L) else 0}")
if len(V):
    kp.gambar_langkah(d, LATIH / "kurva_pelatihan.png")
    plt.show()
    t = V[["langkah", "epoch", "WER", "loss_validasi", "S", "D", "I", "N", "detik_eval", "terbaik"]].copy()
    t["WER"] = (100 * t.WER).round(2)
    print(t.round(3).tail(10).to_string(index=False))

## 2. Penggabungan dan konversi
Adapter checkpoint terbaik digabung ke bobot dasar dengan baris kedua Persamaan 2.12, W = W0 + (alpha/r) B A, dalam float32, lalu disimpan float16 dan dikonversi ke CTranslate2 float16, format yang sama dengan model Tahap 1 dan 2 (`src/konversi_lora.py`). Hasil gabung tiap modul dicocokkan dengan W0 + (alpha/r) B A yang dihitung sendiri.

Model dasar ikut dikonversi dengan langkah yang sama sebagai kontrol. T1-E memakai model CTranslate2 rilisan Systran, sehingga bila konversi sendiri berbeda dari rilisan itu, sebagian selisih T3 terhadap T1-E dapat berasal dari konversi, bukan dari LoRA. Kontrol ini diperiksa di Bagian 3.

In [ ]:
JALANKAN_KONVERSI = True

ADAPTER = LATIH / "adapter_terbaik"
terbaik = json.loads((ADAPTER / "terbaik.json").read_text(encoding="utf-8"))
COMPUTE_TYPE = meta_t1["compute_type"]
KONVERSI = {"T3":    {"adapter": rel(ADAPTER), "hf": MODEL / "hf_gabung", "ct2": MODEL / "ct2_t3"},
            "dasar": {"adapter": None,         "hf": MODEL / "hf_dasar",  "ct2": MODEL / "ct2_dasar"}}
P_KONVERSI = T3 / "konversi.json"
hasil_konversi = json.loads(P_KONVERSI.read_text(encoding="utf-8")) if P_KONVERSI.exists() else {}
print(f"Checkpoint terbaik: langkah {terbaik['langkah']} (epoch {terbaik['epoch']:.2f}), "
      f"WER validasi {100 * terbaik['WER']:.2f}%")

if JALANKAN_KONVERSI:
    for nama, k in KONVERSI.items():
        cfg = {"nama": nama, "model_id": CFG_LATIH["model_id"], "adapter": k["adapter"],
               "hf_keluaran": rel(k["hf"]), "ct2_keluaran": rel(k["ct2"]), "quantization": COMPUTE_TYPE}
        hasil_konversi[nama] = tr.jalankan_terpisah(cfg, KERJA / f"konversi_{nama}.json", cwd=ROOT,
                                                    modul="src.konversi_lora")
        if k["adapter"]:
            hasil_konversi[nama]["langkah_adapter"] = terbaik["langkah"]
    P_KONVERSI.write_text(json.dumps(hasil_konversi, indent=1), encoding="utf-8")
else:
    print("JALANKAN_KONVERSI False: memakai", rel(P_KONVERSI))

assert hasil_konversi.get("T3", {}).get("langkah_adapter") == terbaik["langkah"], \
    "model hasil konversi bukan dari checkpoint terbaik yang sekarang; jalankan ulang konversi"
h = hasil_konversi["T3"]
assert h["skala_alpha_per_r"] == [LORA["alpha"] / LORA["r"]] and h["selisih_gabung_maks"] < 1e-5, h
assert h["n_mels"] == 128 and hasil_konversi["dasar"]["n_mels"] == 128
print(pd.DataFrame(hasil_konversi).T.drop(columns=["adapter"]).to_string())

## 3. Pemeriksaan kesetaraan konversi
Gambar 4.8 dan subbab 3.6.8 butir c. Semua model mentranskripsi 149 segmen validasi dengan kondisi yang sama: float16, greedy, tanpa timestamp (`src/evaluasi_segmen.py`).
- **Kriteria**: adapter terbaik tanpa digabung, yaitu kondisi saat checkpoint dipilih, dibandingkan dengan model CTranslate2 T3. Setara bila selisih WER < 0,5 poin. Persentase segmen yang transkripnya identik ikut dilaporkan.
- **Rincian**: model gabung transformers ditaruh di tengah, supaya selisih yang ada terlihat muncul di penggabungan atau di konversi.
- **Kontrol**: model dasar hasil konversi sendiri dibandingkan dengan model Systran yang dipakai T1-E, dengan ambang yang sama.

Bila belum setara, konversi diperiksa dan Bagian 2 diulang. Data uji tidak dijalankan sebelum kriteria dan kontrol terpenuhi.

In [ ]:
JALANKAN_KESETARAAN = True
AMBANG_SETARA = 0.5   # poin persentase

KES = T3 / "kesetaraan"
KES.mkdir(parents=True, exist_ok=True)
_, val_ref, norm_val = ll.muat_segmen(CFG_LATIH, root=ROOT)
SEG_VAL = [{"seg_latih_id": i, "path": str(SEG_AUDIO / "validasi" / f"{i}.wav")} for i in val_ref.seg_latih_id]
MODEL_EVAL = {
    "peft_terbaik":      {"mode": "hf",  "model": CFG_LATIH["model_id"], "adapter": str(ADAPTER)},
    "hf_gabung":         {"mode": "hf",  "model": str(MODEL / "hf_gabung")},
    "ct2_t3":            {"mode": "ct2", "model": str(MODEL / "ct2_t3")},
    "ct2_dasar_sendiri": {"mode": "ct2", "model": str(MODEL / "ct2_dasar")},
    "ct2_dasar_systran": {"mode": "ct2", "model": VARIAN},   # model yang sama dengan T1-E
}
if JALANKAN_KESETARAAN:
    for nama, m in MODEL_EVAL.items():
        tr.jalankan_terpisah(m | {"nama": nama, "perangkat": "cuda", "compute_type": COMPUTE_TYPE, "segmen": SEG_VAL,
                                  "eval_batch": CFG_LATIH["eval_batch"], "maks_token_baru": CFG_LATIH["maks_token_baru"],
                                  "keluaran": str(KES / f"{nama}.jsonl")},
                             KERJA / f"evaluasi_{nama}.json", cwd=ROOT, modul="src.evaluasi_segmen")
else:
    print("JALANKAN_KESETARAAN False: memakai hipotesis di", rel(KES))

nilai, per_seg = {}, {}
for nama in MODEL_EVAL:
    hyp = {b["seg_latih_id"]: b["hipotesis"] for b in ll.baca_jsonl(KES / f"{nama}.jsonl")}
    assert set(hyp) == set(val_ref.seg_latih_id), f"{nama}: hipotesis tidak lengkap"
    nilai[nama], baris = ll.nilai(val_ref, norm_val, [hyp[i] for i in val_ref.seg_latih_id])
    per_seg[nama] = pd.DataFrame(baris).set_index("seg_latih_id")

val_log = pd.DataFrame(ll.baca_jsonl(LATIH / "log_validasi.jsonl")).set_index("langkah")
model_t = pd.DataFrame(nilai).T[["WER", "S", "D", "I", "N"]]
model_t.loc["log: langkah 0 (dasar, transformers)"] = val_log.loc[0, ["WER", "S", "D", "I", "N"]]
model_t.loc[f"log: langkah {terbaik['langkah']} (terbaik)"] = val_log.loc[terbaik["langkah"], ["WER", "S", "D", "I", "N"]]
model_t["WER"] = (100 * model_t.WER.astype(float)).round(2)
model_t.to_csv(KES / "kesetaraan_model.csv", encoding="utf-8-sig")
print("WER validasi tiap model (%):")
print(model_t.to_string())

PASANGAN = [("peft_terbaik", "ct2_t3", "kriteria"), ("peft_terbaik", "hf_gabung", "rincian: penggabungan"),
            ("hf_gabung", "ct2_t3", "rincian: konversi"), ("ct2_dasar_systran", "ct2_dasar_sendiri", "kontrol")]
baris = []
for a, b, peran in PASANGAN:
    sa, sb = per_seg[a], per_seg[b].loc[per_seg[a].index]
    baris.append({"model_A": a, "model_B": b, "peran": peran,
                  "WER_A": 100 * nilai[a]["WER"], "WER_B": 100 * nilai[b]["WER"],
                  "selisih_poin": 100 * (nilai[b]["WER"] - nilai[a]["WER"]),
                  "identik_ternormalisasi_%": 100 * (sa.hipotesis_norm == sb.hipotesis_norm).mean(),
                  "identik_mentah_%": 100 * (sa.hipotesis == sb.hipotesis).mean()})
kes = pd.DataFrame(baris)
kes["setara"] = np.where(kes.peran.isin(["kriteria", "kontrol"]), kes.selisih_poin.abs() < AMBANG_SETARA, None)
kes.round(3).to_csv(KES / "kesetaraan.csv", index=False, encoding="utf-8-sig")
pd.concat({n: d[["hipotesis", "hipotesis_norm", "S", "D", "I", "N"]] for n, d in per_seg.items()}, axis=1) \
  .to_csv(KES / "kesetaraan_segmen.csv", encoding="utf-8-sig")
print()
print(kes.round(2).to_string(index=False))

SETARA = bool(kes.loc[kes.peran == "kriteria", "setara"].iloc[0])
KONTROL = bool(kes.loc[kes.peran == "kontrol", "setara"].iloc[0])
selisih_log = 100 * (nilai["peft_terbaik"]["WER"] - val_log.loc[terbaik["langkah"], "WER"])
print(f"\nAdapter terbaik di perangkat ini vs log pelatihan: selisih {selisih_log:+.2f} poin")
print(f"Konversi setara: {SETARA}; kontrol konversi setara: {KONTROL}")
if not (SETARA and KONTROL):
    print("[!] Belum setara. Periksa konversi lalu ulangi Bagian 2 (Gambar 4.8). Data uji tidak dijalankan.")

## 4. Pemeriksaan pada rekaman validasi utuh
Model dilatih tanpa timestamp, sedangkan pengujian memakai parameter Tabel 3.5 dengan mode timestamp atas rekaman utuh. Sebelum menyentuh data uji, T3 dan model dasar (sama dengan T1-E) dijalankan atas 4 rekaman validasi utuh dengan parameter Tabel 3.5, satu ulangan, lalu dihitung dengan prosedur subbab 3.6.8. Percobaan awal memakai rekaman validasi terpendek, bukan rekaman uji. Hasilnya dilaporkan apa adanya dan tidak dipakai untuk melatih ulang.

In [ ]:
JALANKAN_VALIDASI_UTUH = True

PARAMETER = meta_t1["parameter_decoding"]
ULANGAN = meta_t1["rtf"]["ulangan"]
PEMANASAN = meta_t1["rtf"]["pemanasan"]
DEVICE = "cuda"
assert "hotwords" not in PARAMETER and PARAMETER["condition_on_previous_text"] is False

VU_DIR = T3 / "validasi_utuh"
LOG_VU = VU_DIR / "waktu_transkripsi.jsonl"
KONF_VU = {"dasar": VARIAN, "T3": str(MODEL / "ct2_t3")}
durasi_val = {}
for f in validasi:
    p = AUDIO_PROC / f"{f}.wav"
    assert p.exists(), f"WAV rekaman validasi {f} tidak ditemukan di {rel(AUDIO_PROC)}"
    i = sf.info(str(p))
    assert (i.samplerate, i.channels) == (16000, 1), f"{f}: {i.samplerate} Hz, {i.channels} kanal"
    durasi_val[f] = i.frames / i.samplerate
fp_val = min(durasi_val, key=durasi_val.get)

if JALANKAN_VALIDASI_UTUH:
    assert SETARA, "konversi belum setara (Bagian 3)"
    for konf, model in KONF_VU.items():
        cfg = {"nama": konf, "model": model, "device": DEVICE, "compute_type": COMPUTE_TYPE,
               "parameter": PARAMETER, "ulangan": 1,
               "rekaman": [{"file_id": f, "path": str(AUDIO_PROC / f"{f}.wav"), "durasi": round(durasi_val[f], 3)}
                           for f in validasi],
               "pemanasan": {"path": str(AUDIO_PROC / f"{fp_val}.wav"), "detik": PEMANASAN["detik"]},
               "arsip_mentah": str(VU_DIR / "mentah" / konf), "log_waktu": str(LOG_VU)}
        tr.jalankan_terpisah(cfg, KERJA / f"validasi_utuh_{konf}.json", cwd=ROOT)
else:
    print("JALANKAN_VALIDASI_UTUH False: memakai hasil yang sudah ada di", rel(VU_DIR))

waktu_vu = pd.DataFrame(ll.baca_jsonl(LOG_VU))
acuan_val = {f: json.loads((ACUAN_DIR / f"{f}.json").read_text(encoding="utf-8")) for f in validasi}
norm_vu = {f: pt.Normalisasi(kamus, glosarium[acuan_val[f]["mata_kuliah"]]) for f in validasi}
hasil_vu = [ld.log_konfigurasi(k, k, VU_DIR / "mentah" / k, acuan_val, norm_vu, durasi_val, manifest,
                               waktu_vu[waktu_vu.konfigurasi == k], kanonik) for k in KONF_VU]
assert all(h["verifikasi"].sama.all() for h in hasil_vu), "WER penyelarasan berbeda dengan jiwer"
ld.tulis_log(hasil_vu, VU_DIR / "log")

rek_vu = an.metrik_dari_jumlah(pd.concat([h["rekaman"] for h in hasil_vu], ignore_index=True))
p = rek_vu.pivot_table(index="file_id", columns="kode", values=an.METRIK)
agr = an.gabung(rek_vu, "kode")
t = pd.DataFrame({f"{m}_{k}": 100 * p[m][k] for m in an.METRIK for k in KONF_VU})
for m in an.METRIK:
    t[f"selisih_{m}"] = t[f"{m}_T3"] - t[f"{m}_dasar"]
t.loc["agregat"] = [100 * agr.at[k, m] for m in an.METRIK for k in KONF_VU] + \
                   [100 * (agr.at["T3", m] - agr.at["dasar", m]) for m in an.METRIK]
t.insert(0, "jalur_acuan", [manifest.jalur_acuan.get(f, "") for f in t.index])
t.round(2).to_csv(VU_DIR / "metrik_rekaman.csv", encoding="utf-8-sig")
print("Rekaman validasi utuh, parameter Tabel 3.5 (persen; selisih dalam poin, T3 dikurangi dasar):")
print(t.round(2).to_string())
seg_vu = rek_vu.set_index(["kode", "file_id"])[["segmen", "segmen_dibuang"]].unstack("kode")
print("\nSegmen keluaran dan yang dibuang karena di luar acuan:")
print(seg_vu.to_string())

## 5. Transkripsi T3
Model CTranslate2 T3, presisi, parameter decoding, tiga ulangan, dan percobaan awal sama dengan Tahap 1, tanpa hotwords. Satu-satunya perbedaan dengan T1-E adalah bobot model. Seluruh rekaman dikerjakan dalam satu proses (`src/transkripsi.py`).

In [ ]:
JALANKAN_TRANSKRIPSI = True

audio_ada = {f: (AUDIO_PROC / f"{f}.wav").exists() for f in uji}
waktu_t1 = pd.DataFrame(ll.baca_jsonl(T1 / "waktu_transkripsi.jsonl"))
durasi_t1 = waktu_t1.groupby("file_id").durasi_s.first()
durasi = {}
for f in uji:
    if audio_ada[f]:
        i = sf.info(str(AUDIO_PROC / f"{f}.wav"))
        assert (i.samplerate, i.channels) == (16000, 1), f"{f}: {i.samplerate} Hz, {i.channels} kanal"
        durasi[f] = i.frames / i.samplerate
    else:
        durasi[f] = float(durasi_t1[f])
    assert abs(durasi[f] - durasi_t1[f]) < 0.01, f"{f}: durasi berbeda dengan Tahap 1"
MODEL_T3 = str(MODEL / "ct2_t3")

if JALANKAN_TRANSKRIPSI:
    assert SETARA and KONTROL, "Pemeriksaan kesetaraan (Bagian 3) belum lolos. Data uji tidak boleh disentuh."
    assert all(audio_ada.values()), f"WAV tidak ditemukan: {[f for f, a in audio_ada.items() if not a]}"
    print(f"Audio yang diproses: {sum(durasi.values())/3600:.2f} jam x {ULANGAN} ulangan")
    print(f"VRAM terpakai sebelum mulai: {nvsmi('memory.used')} MB\n")
    fp = PEMANASAN["file_id"]
    cfg = {"nama": KODE, "model": MODEL_T3, "device": DEVICE, "compute_type": COMPUTE_TYPE,
           "parameter": PARAMETER, "ulangan": ULANGAN,
           "rekaman": [{"file_id": f, "path": str(AUDIO_PROC / f"{f}.wav"), "durasi": round(durasi[f], 3)} for f in uji],
           "pemanasan": {"path": str(AUDIO_PROC / f"{fp}.wav"), "detik": PEMANASAN["detik"]},
           "arsip_mentah": str(MENTAH), "log_waktu": str(LOG_WAKTU)}
    tr.jalankan_terpisah(cfg, KERJA / "transkripsi_t3.json", cwd=ROOT)
else:
    print("JALANKAN_TRANSKRIPSI False: memakai hasil yang sudah ada di", rel(T3))

In [ ]:
assert LOG_WAKTU.exists(), "Belum ada hasil transkripsi T3. Jalankan sel sebelumnya dengan JALANKAN_TRANSKRIPSI = True."
waktu = pd.DataFrame(ll.baca_jsonl(LOG_WAKTU))
waktu = waktu[(waktu.konfigurasi == KODE) & waktu.file_id.isin(uji)]
assert len(waktu) == ULANGAN * len(uji), f"baru {len(waktu)} dari {ULANGAN * len(uji)} pasangan ulangan x rekaman"

# Keluaran mentah harus lengkap, dari model T3, dan tanpa hotwords
for f in uji:
    p = MENTAH / f"{f}.json"
    assert p.exists(), f"keluaran mentah {f} tidak ada"
    m = json.loads(p.read_text(encoding="utf-8"))
    assert Path(m["model"]).name == "ct2_t3" and "hotwords" not in m["parameter"], f"{f}: model atau parameter berbeda"

sidik = waktu.pivot_table(index="file_id", columns="ulangan", values="sidik", aggfunc="first")
beda = sidik[sidik.nunique(axis=1) > 1]
print(f"{len(waktu)} pasangan ulangan x rekaman tercatat; keluaran berbeda antarulangan: {len(beda)} dari {len(sidik)} rekaman")
if len(beda):
    print(beda.to_string())

## 6. Metrik dan log per data
Prosedur subbab 3.6.8, sama dengan Tahap 1 dan 2. `src/log_data.py` menghitung metrik sekaligus menulis log per rekaman, segmen, istilah, dan kata ke `tahap3/log/`, dengan format yang sama dengan `tahap1/log/` dan `tahap2/log/`.

In [ ]:
acuan = {f: json.loads((ACUAN_DIR / f"{f}.json").read_text(encoding="utf-8")) for f in uji}
MK = {f: acuan[f]["mata_kuliah"] for f in uji}
norm = {f: pt.Normalisasi(kamus, glosarium[MK[f]]) for f in uji}

hasil = ld.log_konfigurasi(KODE, KODE, MENTAH, acuan, norm, durasi, manifest, waktu, kanonik)
blok = hasil["blok"]
blok.to_csv(T3 / "metrik_blok.csv", index=False, encoding="utf-8-sig")
hasil["verifikasi"].to_csv(T3 / "verifikasi_jiwer.csv", index=False, encoding="utf-8-sig")
assert hasil["verifikasi"].sama.all(), "WER penyelarasan berbeda dengan jiwer"
assert len(blok) == len(pd.read_csv(T1 / "metrik_blok.csv").query("kode == @KODE_DASAR")), "jumlah blok berbeda dengan baseline"

jumlah = ld.tulis_log([hasil], T3 / "log")
print(f"WER sama dengan jiwer: {hasil['verifikasi'].sama.sum()} dari {len(uji)} rekaman; {len(blok)} blok")
print("Log ditulis ke", rel(T3 / "log"), jumlah)

rek = hasil["rekaman"]
t = rek.set_index("file_id")[["mata_kuliah", "jalur_acuan", "segmen", "segmen_dibuang", "istilah_salah",
                              "istilah_muncul", "WER", "CER", "TTER", "rtf_rerata"]].copy()
t[["WER", "CER", "TTER"]] = (100 * t[["WER", "CER", "TTER"]]).round(2)
t["rtf_rerata"] = t.rtf_rerata.round(4)
print("\nPer rekaman (WER, CER, TTER dalam persen):")
print(t.to_string())

In [ ]:
# Agregat T3 dan RTF, dengan kolom yang sama seperti tahap1/metrik_varian.csv dan tahap2/metrik_t2.csv
rtf_u = waktu.groupby("ulangan").agg(proses=("proses_s", "sum"), durasi=("durasi_s", "sum"))
rtf_u["rtf"] = rtf_u.proses / rtf_u.durasi
rtf_u.reset_index().assign(konfigurasi=KODE).to_csv(T3 / "rtf_ulangan.csv", index=False, encoding="utf-8-sig")
rtf_rek = waktu.groupby("file_id").rtf.mean()
sesi_tr = waktu.sort_values("waktu").iloc[0]

agregat = an.gabung(blok, "kode")
agregat.insert(0, "varian", VARIAN)
agregat = agregat.assign(RTF=rtf_u.rtf.mean(), RTF_sd=rtf_u.rtf.std(ddof=1),
                         RTF_min_rekaman=rtf_rek.min(), RTF_maks_rekaman=rtf_rek.max(),
                         muat_model_s=sesi_tr.muat_s, vram_puncak_mb=sesi_tr.vram_puncak_mb - sesi_tr.vram_awal_mb)
agregat.to_csv(T3 / "metrik_t3.csv", encoding="utf-8-sig")

dasar = pd.read_csv(T1 / "metrik_varian.csv", encoding="utf-8-sig").set_index("kode").loc[[KODE_DASAR]]
t2 = pd.read_csv(T2 / "metrik_t2.csv", encoding="utf-8-sig", index_col=0).loc[[KODE_T2]]
URUT = [KODE_DASAR, KODE_T2, KODE]
banding = pd.concat([dasar, t2, agregat])[["WER", "CER", "TTER", "RTF", "RTF_sd", "vram_puncak_mb"]].loc[URUT]
banding.loc[f"selisih vs {KODE_DASAR}"] = banding.loc[KODE] - banding.loc[KODE_DASAR]
banding.loc[f"selisih vs {KODE_T2}"] = banding.loc[KODE] - banding.loc[KODE_T2]
banding[["WER", "CER", "TTER"]] = (100 * banding[["WER", "CER", "TTER"]]).round(2)
banding[["RTF", "RTF_sd"]] = banding[["RTF", "RTF_sd"]].round(4)
print("Agregat (WER, CER, TTER dalam persen; selisih dalam poin persentase):")
print(banding.to_string())

## 7. Analisis terhadap T1-E dan T2
Subbab 3.4 dan Tabel 3.7: T3 dibandingkan dengan baseline T1-E dan dengan T2. Kedua pembanding dibaca dari `tahap1/` dan `tahap2/` tanpa dihitung ulang, dan seluruh perbandingan berpasangan atas rekaman dan blok yang sama.

In [ ]:
blok_dasar = (pd.read_csv(T1 / "metrik_blok.csv", encoding="utf-8-sig").query("kode == @KODE_DASAR")
                .rename(columns={"varian": "konfigurasi"}))
blok_t2 = pd.read_csv(T2 / "metrik_blok.csv", encoding="utf-8-sig").query("kode == @KODE_T2")
# metrik dihitung ulang dari bilangan bulat supaya pembulatan CSV tidak terbaca sebagai selisih
blok_gab = an.metrik_dari_jumlah(pd.concat([blok_dasar, blok_t2, blok], ignore_index=True))
rek_dasar = pd.read_csv(T1 / "log" / "rekaman.csv", encoding="utf-8-sig").query("kode == @KODE_DASAR")
rek_t2 = pd.read_csv(T2 / "log" / "rekaman.csv", encoding="utf-8-sig").query("kode == @KODE_T2")
rek_gab = an.metrik_dari_jumlah(pd.concat([rek_dasar, rek_t2, rek], ignore_index=True))
baca_kem = lambda p, k: (pd.read_csv(p, keep_default_na=False, encoding="utf-8-sig").query("konfigurasi == @k"))
KEM = {KODE_DASAR: baca_kem(T1 / "log" / "kemunculan_istilah.csv", VARIAN),
       KODE_T2: baca_kem(T2 / "log" / "kemunculan_istilah.csv", KODE_T2)}
PEMBANDING = [KODE_DASAR, KODE_T2]
for k, b in ((KODE_DASAR, blok_dasar), (KODE_T2, blok_t2)):
    print(f"{k}: {len(b)} blok, {len(KEM[k])} kemunculan istilah")

### 7.1 Statistik deskriptif dan ambang
Format: agregat (rerata blok ± sd; min-maks). WER dibandingkan dengan ambang 25 persen (Munteanu, et al., 2006) dan RTF dengan ambang 1 (Pratap, et al., 2020).

In [ ]:
desk = an.deskriptif(blok_gab, "kode").loc[URUT]
desk = desk.join(pd.DataFrame({"RTF_rerata": banding.loc[URUT, "RTF"], "RTF_sd": banding.loc[URUT, "RTF_sd"]}))
desk.round(4).to_csv(T3 / "statistik_deskriptif.csv", encoding="utf-8-sig")

ringkas = lambda m: desk.apply(lambda r: f"{r[f'{m}_agregat']:.2f} ({r[f'{m}_rerata']:.2f} ± {r[f'{m}_sd']:.2f}; "
                                          f"{r[f'{m}_min']:.2f}-{r[f'{m}_maks']:.2f})", axis=1)
print(pd.DataFrame({f"{m} %": ringkas(m) for m in an.METRIK}
                   | {"RTF": desk.apply(lambda r: f"{r.RTF_rerata:.4f} ± {r.RTF_sd:.4f}", axis=1)}).to_string())

ambang = pd.DataFrame({
    "WER %": desk.WER_agregat.round(2),
    "WER <= 25%": desk.WER_agregat <= 25,
    "blok dengan WER <= 25%": blok_gab.groupby("kode").WER.apply(lambda s: f"{(s <= 0.25).sum()}/{s.notna().sum()}").loc[URUT],
    "RTF": desk.RTF_rerata.round(4),
    "RTF <= 1": desk.RTF_rerata <= 1,
})
ambang.to_csv(T3 / "ambang.csv", encoding="utf-8-sig")
print()
print(ambang.to_string())

### 7.2 Uji Wilcoxon T3 terhadap T1-E dan T2
Tingkat blok (n = 54, TTER 53) dengan ambang p < 0,001 sebagai dasar kesimpulan, dan tingkat konfigurasi atas nilai tiap rekaman (n = 9) sebagai pelengkap. `selisih_median` adalah T3 dikurangi pembanding dalam poin persentase; nilai negatif berarti T3 lebih rendah kesalahannya.

In [ ]:
ALFA_BLOK = meta_t1["metrik"]["ambang_p_blok"]
baris = []
for B in PEMBANDING:
    for tingkat, data, idx in (("blok", blok_gab, ["file_id", "blok"]), ("konfigurasi", rek_gab, ["file_id"])):
        for m in an.METRIK:
            tab = data.pivot_table(index=idx, columns="kode", values=m)[[KODE, B]].dropna()
            baris.append({"tingkat": tingkat, "metrik": m, "konfigurasi_A": KODE, "konfigurasi_B": B,
                          **an.uji_wilcoxon(tab[KODE], tab[B])})
wil = pd.DataFrame(baris)
wil["signifikan"] = np.where(wil.tingkat == "blok", np.where(wil.p < ALFA_BLOK, "ya", "tidak"), "-")
wil.to_csv(T3 / "wilcoxon.csv", index=False, encoding="utf-8-sig")

for B in PEMBANDING:
    w = wil[wil.konfigurasi_B == B]
    t = w.pivot_table(index="metrik", columns="tingkat", values="p", sort=False)
    b = w[w.tingkat == "blok"].set_index("metrik")
    t = t.assign(selisih_median=b.selisih_median.round(2), n_tak_nol=b.n_tak_nol, signifikan=b.signifikan)
    print(f"{KODE} terhadap {B}:")
    print(t[["selisih_median", "n_tak_nol", "blok", "konfigurasi", "signifikan"]]
          .to_string(formatters={"blok": "{:.2e}".format, "konfigurasi": "{:.4f}".format}))
    print()

### 7.3 Perubahan WER dan TTER per rekaman
Disajikan berdampingan (Ugan, et al., 2025) supaya terlihat rekaman yang kedua metriknya bergerak berlawanan arah. Selisih adalah T3 dikurangi pembanding, dalam poin persentase.

In [ ]:
p = rek_gab.pivot_table(index="file_id", columns="kode", values=["WER", "TTER"])
info = rek.set_index("file_id")
ubah = pd.DataFrame({"mata_kuliah": info.mata_kuliah, "jalur_acuan": info.jalur_acuan})
for m in ("WER", "TTER"):
    for k in URUT:
        ubah[f"{m}_{k}"] = 100 * p[m][k]
    for B in PEMBANDING:
        ubah[f"selisih_{m}_vs_{B}"] = 100 * (p[m][KODE] - p[m][B])
for B in PEMBANDING:
    ubah[f"arah_vs_{B}"] = np.where(np.sign(ubah[f"selisih_WER_vs_{B}"]) * np.sign(ubah[f"selisih_TTER_vs_{B}"]) < 0,
                                    "berlawanan", "")
ubah = ubah.loc[uji]
ubah.round(2).to_csv(T3 / "perubahan_rekaman.csv", encoding="utf-8-sig")
print(ubah.round(2).to_string())

### 7.4 Rincian per jalur acuan
Acuan jalur whisper, termasuk data latih, disusun dari keluaran large-v3 lalu dikoreksi, sehingga baseline di jalur itu mendekati batas atas dan setiap perubahan keluaran cenderung terhitung sebagai kesalahan baru. Pada T3 efek ini dua arah: model juga dilatih dengan acuan bergaya large-v3. Tabel ini deskriptif, hanya untuk membantu menafsirkan Bagian 7.1 dan 7.2.

In [ ]:
j = an.gabung(rek_gab, ["jalur_acuan", "kode"])[an.METRIK].mul(100).unstack("kode")
for m in an.METRIK:
    for B in PEMBANDING:
        j[(m, f"selisih vs {B}")] = j[(m, KODE)] - j[(m, B)]
j = j[[(m, k) for m in an.METRIK for k in URUT + [f"selisih vs {B}" for B in PEMBANDING]]]
j["rekaman"] = rek.groupby("jalur_acuan").size()
j["segmen_latih"] = seg[seg.split == "latih"].merge(manifest.jalur_acuan, left_on="file_id", right_index=True) \
                       .groupby("jalur_acuan").size()
j.round(2).to_csv(T3 / "per_jalur.csv", encoding="utf-8-sig")
print(j.round(2).to_string())

### 7.5 Perubahan per istilah
Setiap kemunculan istilah pada acuan dipasangkan antara pembanding dan T3, lalu digolongkan menjadi *tetap benar*, *diperbaiki*, *dirusak*, atau *tetap salah*. Kolom `di_data_latih` menandai entri glosarium yang muncul di transkrip segmen latih mata kuliahnya, sebagai padanan kolom `di_hotwords` pada Tahap 2.

In [ ]:
# Entri glosarium yang muncul di label segmen latih, per mata kuliah
muncul_latih = {}
for r in seg[seg.split == "latih"].itertuples():
    tok = pt.Normalisasi(kamus, glosarium[r.mata_kuliah])(r.teks)
    for _, _, e in pt.tandai_glosarium(tok, glosarium[r.mata_kuliah]):
        muncul_latih[(r.mata_kuliah, e)] = muncul_latih.get((r.mata_kuliah, e), 0) + 1

pi = pd.concat([an.bandingkan_istilah(KEM[B], hasil["istilah"]).assign(pembanding=B) for B in PEMBANDING],
               ignore_index=True)
pi["kemunculan_di_latih"] = [muncul_latih.get((mk, e), 0) for mk, e in zip(pi.mata_kuliah, pi.entri_id)]
pi["di_data_latih"] = pi.kemunculan_di_latih > 0
pi.to_csv(T3 / "perubahan_istilah.csv", index=False, encoding="utf-8-sig")

print("Kemunculan istilah menurut status:")
print(pd.crosstab([pi.pembanding, pi.di_data_latih], pi.status, margins=True, margins_name="total").to_string())

per_entri = (pi.groupby(["pembanding", "mata_kuliah", "entri_id", "bentuk_kanonik", "di_data_latih"]).status
               .value_counts().unstack(fill_value=0).reset_index())
for s in ("diperbaiki", "dirusak", "tetap salah", "tetap benar"):
    if s not in per_entri:
        per_entri[s] = 0
per_entri.to_csv(T3 / "perubahan_istilah_entri.csv", index=False, encoding="utf-8-sig")
kolom = ["mata_kuliah", "bentuk_kanonik", "di_data_latih", "diperbaiki", "dirusak", "tetap salah"]
for B in PEMBANDING:
    for s in ("diperbaiki", "dirusak"):
        t = per_entri[(per_entri.pembanding == B) & (per_entri[s] > 0)].sort_values(s, ascending=False).head(10)
        print(f"\nEntri yang paling banyak {s} (terhadap {B}):")
        print(t[kolom].to_string(index=False) if len(t) else "  tidak ada")

### 7.6 Ekspor kesalahan istilah dan kandidat halusinasi untuk dilabeli
Subbab 3.4 butir 5, dengan aturan yang sama seperti Tahap 1 dan 2, diturunkan dari log Bagian 6. Kolom `golongan` diisi `substitusi_mirip`, `penghapusan`, `transliterasi`, atau `lainnya`; kolom `halusinasi` diisi `ya` atau `tidak`. Berkas yang sudah mulai dilabeli tidak ditimpa.

In [ ]:
def sudah_dilabeli(p, kolom):
    if not p.exists():
        return False
    lama = pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    return bool((lama[kolom].ne("") & lama[kolom].ne("penghapusan")).any() or lama.catatan.ne("").any())

for berkas, df, kolom in (("kesalahan_istilah.csv", an.kesalahan_istilah(hasil["istilah"]), "golongan"),
                          ("kandidat_halusinasi.csv", an.kandidat_halusinasi(T3 / "log", KODE), "halusinasi")):
    p = T3 / berkas
    if sudah_dilabeli(p, kolom):
        print(f"{berkas}: sudah berisi label, tidak ditimpa")
        continue
    df.to_csv(p, index=False, encoding="utf-8-sig")
    rinci = (df.sumber.value_counts().to_dict() if "sumber" in df
             else {"terisi otomatis": int((df.golongan != "").sum()), "perlu dilabeli": int((df.golongan == "").sum())})
    print(f"{berkas}: {len(df)} baris {rinci}")

### 7.7 Kurva pelatihan per epoch
Loss latih dirata-rata per epoch dan dibandingkan dengan loss validasi (teacher forcing, rerata per segmen) pada sumbu yang sama, lalu WER validasi diletakkan pada posisi epochnya. Kurva ini untuk membaca apakah pelatihan sudah konvergen: loss latih yang melandai, loss validasi yang mendatar atau naik kembali (tanda overfitting), dan letak checkpoint terbaik. Laju belajar 1e-5 tergolong kecil untuk LoRA (subbab 3.6.7), sehingga kurva ini juga menunjukkan apakah 35 epoch cukup. Ringkasan konvergensi bersifat deskriptif dan tidak mengubah checkpoint yang sudah dipilih.

In [ ]:
d = kp.muat(LATIH)
ep = kp.per_epoch(d)
ep.round(5).to_csv(T3 / "pelatihan_per_epoch.csv", encoding="utf-8-sig")
kp.gambar_epoch(d, T3 / "kurva_per_epoch.png")
plt.show()
kolom = ["loss_latih", "perubahan_loss_%", "langkah_validasi", "WER_validasi", "loss_validasi"]
print(ep[kolom].round(4).to_string())
konvergensi = kp.ringkas_konvergensi(d)
konvergensi.to_csv(T3 / "konvergensi.csv", header=["nilai"], encoding="utf-8-sig")
print()
print(konvergensi.to_string())

## 8. Sumber daya Tahap 3
Waktu pelatihan, puncak memori GPU pelatihan, ukuran adapter, dan RTF T3. Perangkat pelatihan dicatat per sesi; bila pelatihan dijalankan di server, angka pelatihan berasal dari server, sedangkan RTF dan memori inferensi berasal dari perangkat Tabel 3.2.

In [ ]:
from safetensors import safe_open
log_langkah = pd.DataFrame(ll.baca_jsonl(LATIH / "log_langkah.jsonl"))
log_val = pd.DataFrame(ll.baca_jsonl(LATIH / "log_validasi.jsonl"))
sesi = pd.DataFrame(ll.baca_jsonl(LATIH / "sesi.jsonl"))
with safe_open(str(ADAPTER / "adapter_model.safetensors"), "pt") as fh:
    n_lora = sum(fh.get_tensor(k).numel() for k in fh.keys())
kelayakan = pd.read_csv(T1 / "uji_kelayakan.csv", encoding="utf-8-sig").set_index("varian").loc[VARIAN]
ukuran = lambda d: sum(f.stat().st_size for f in Path(d).rglob("*") if f.is_file()) / 2**20

sumber = pd.Series({
    "perangkat pelatihan": ", ".join(sorted(sesi.gpu.dropna().unique())) if "gpu" in sesi else "",
    "host pelatihan": ", ".join(sorted(sesi.host.unique())),
    "jumlah sesi pelatihan": len(sesi),
    "durasi sesi total (jam)": round(sesi.durasi_jam.sum(), 2),
    "waktu langkah latih (jam)": round(log_langkah.detik.sum() / 3600, 2),
    "waktu validasi (jam)": round(log_val.detik_eval.sum() / 3600, 2),
    "jumlah validasi": len(log_val),
    "detik per langkah optimizer (median)": round(log_langkah.detik.median(), 3),
    "detik per batch maju-mundur (median)": round(log_langkah.detik.median() / CFG_LATIH["akumulasi"], 3),
    "uji kelayakan: detik per batch maju-mundur": kelayakan.detik_per_langkah,
    "puncak VRAM pelatihan dicadangkan (MB)": int(log_val.vram_puncak_dicadangkan_mb.max())
                                               if "vram_puncak_dicadangkan_mb" in log_val else None,
    "uji kelayakan: puncak VRAM (MB)": int(kelayakan.puncak_memori_mb),
    "parameter terlatih": n_lora,
    "% terlatih": kelayakan["%_terlatih"],
    "ukuran adapter (MB)": round(ukuran(ADAPTER), 2),
    "ukuran model CTranslate2 T3 (MB)": round(ukuran(MODEL / "ct2_t3"), 1) if (MODEL / "ct2_t3").exists() else None,
    "langkah terbaik": terbaik["langkah"], "epoch terbaik": round(terbaik["epoch"], 2),
    "RTF T3": round(agregat.RTF.iloc[0], 4), "RTF T1-E": round(dasar.RTF.iloc[0], 4),
    "VRAM inferensi T3 (MB)": int(agregat.vram_puncak_mb.iloc[0]),
    "muat model T3 (detik)": float(agregat.muat_model_s.iloc[0]),
})
assert n_lora == kelayakan.parameter_terlatih, "jumlah parameter adapter berbeda dengan uji kelayakan"
sumber.to_csv(T3 / "sumber_daya.csv", header=["nilai"], encoding="utf-8-sig")
print(sumber.to_string())

## 9. Metadata run

In [ ]:
import platform, importlib.metadata as im
def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return None

meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "Tahap 3: fine-tuning LoRA",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": GPU,
    "varian": VARIAN, "kode": KODE, "pembanding": PEMBANDING,
    "pelatihan": {"konfigurasi": CFG_LATIH, "rencana": R, "sidik": konf["sidik"], "terbaik": terbaik,
                  "sesi": ll.baca_jsonl(LATIH / "sesi.jsonl"),
                  "metadata_server": rel(LATIH / "run_meta_latih.json") if (LATIH / "run_meta_latih.json").exists() else None},
    "konversi": hasil_konversi,
    "kesetaraan": {"ambang_poin": AMBANG_SETARA, "setara": SETARA, "kontrol_setara": KONTROL,
                   "pasangan": kes.round(4).to_dict("records")},
    "validasi_utuh": {"rekaman": validasi, "pemanasan": {"file_id": fp_val, "detik": PEMANASAN["detik"]},
                      "ulangan": 1, "berkas": rel(VU_DIR / "metrik_rekaman.csv")},
    "parameter_decoding": PARAMETER, "compute_type": COMPUTE_TYPE,
    "sumber_parameter": rel(T1 / "run_meta_tahap1.json"),
    "rtf": {"ulangan": ULANGAN, "pemanasan": PEMANASAN},
    "metrik": {"panjang_blok_detik": mt.PANJANG_BLOK, "ambang_p_blok": ALFA_BLOK},
    "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
    "versi_pustaka": {p: versi(p) for p in ["faster-whisper", "ctranslate2", "onnxruntime", "tokenizers", "jiwer",
                                             "scipy", "numpy", "pandas", "torch", "transformers", "peft",
                                             "accelerate", "safetensors", "soundfile"]},
}
(T3 / "run_meta_tahap3.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

## Output

| Berkas di `data/output/tahap3/` | Isi | Dipakai di |
|---|---|---|
| `latih/konfigurasi.json` | Konfigurasi Tabel 3.6 yang dikunci beserta rencana langkah | Bab 6, lampiran |
| `latih/log_langkah.jsonl`, `latih/log_validasi.jsonl`, `latih/kurva_pelatihan.png` | Loss latih per langkah optimizer, WER dan loss validasi per evaluasi, dan kurvanya | Bab 6, pemantauan laju belajar |
| `latih/validasi_segmen.jsonl` | Hipotesis tiap segmen validasi pada tiap evaluasi | Penelusuran segmen validasi yang memburuk |
| `latih/sesi.jsonl` | Perangkat, langkah awal dan akhir, durasi tiap sesi pelatihan | Analisis sumber daya |
| `latih/adapter_terbaik/` | Adapter checkpoint terbaik dan `terbaik.json` | Bagian 2 |
| `konversi.json`, `model/` | Hasil pemeriksaan penggabungan dan ukuran model; model gabung dan CTranslate2 (tidak di-commit) | Bagian 3 dan 5, lampiran |
| `kesetaraan/kesetaraan.csv`, `kesetaraan_model.csv`, `kesetaraan_segmen.csv` | WER validasi tiap model, selisih dan persentase identik tiap pasangan, hipotesis per segmen | Bab 6, bukti kesetaraan konversi |
| `validasi_utuh/metrik_rekaman.csv`, `validasi_utuh/log/` | T3 dan model dasar pada 4 rekaman validasi utuh dengan parameter Tabel 3.5 | Bab 6 |
| `mentah/<ID>.json`, `waktu_transkripsi.jsonl`, `rtf_ulangan.csv` | Keluaran mentah ulangan pertama, waktu proses tiap ulangan, RTF | Bab 6, analisis sumber daya |
| `metrik_blok.csv`, `metrik_t3.csv`, `verifikasi_jiwer.csv` | Pembilang, penyebut, WER, CER, TTER per blok dan agregat T3; bukti penyelarasan | Bab 6, lampiran |
| `log/rekaman.csv`, `log/segmen_hipotesis.csv`, `log/kemunculan_istilah.csv`, `log/penyelarasan/T3/<ID>.tsv` | Log per rekaman, segmen, istilah, dan kata, format sama dengan Tahap 1 dan 2 | Penelusuran kesalahan per rekaman, segmen, kata |
| `statistik_deskriptif.csv`, `ambang.csv`, `wilcoxon.csv` | T3 terhadap T1-E dan T2 | Bab 6, rumusan masalah kedua |
| `perubahan_rekaman.csv`, `per_jalur.csv` | Perubahan WER dan TTER per rekaman dan per jalur acuan | Bab 6 |
| `perubahan_istilah.csv`, `perubahan_istilah_entri.csv` | Istilah yang diperbaiki dan dirusak T3 terhadap tiap pembanding, beserta kemunculannya di data latih | Bab 6, analisis kesalahan istilah |
| `kesalahan_istilah.csv`, `kandidat_halusinasi.csv` | Tabel untuk dilabeli manual | Analisis kesalahan istilah (3.4 butir 5) |
| `pelatihan_per_epoch.csv`, `kurva_per_epoch.png`, `konvergensi.csv` | Loss latih dan validasi serta WER validasi per epoch, ringkasan konvergensi | Bab 6, pembahasan konvergensi |
| `sumber_daya.csv`, `run_meta_tahap3.json` | Waktu pelatihan, puncak VRAM, ukuran adapter, RTF; metadata run | Bab 6, analisis sumber daya |